# 🔮 Notebook 3 — Forecasting & Statistical Analysis

We use **linear regression** to predict daily revenue based on load shedding hours,
and calculate the **Pearson correlation** to statistically prove the relationship
between load shedding and revenue loss.

**New concepts covered:**
- Pearson correlation coefficient
- Linear regression with scikit-learn
- R² score — how well the model fits
- Scenario forecasting for April 2024

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv('../data/merged.csv', parse_dates=['date'])
trading = df[(df['is_public_holiday']==False) & (df['revenue']>0)].copy()
print('Trading days loaded:', len(trading))

## Step 1 — Pearson Correlation

The **Pearson correlation coefficient** measures how strongly two variables move together.
- Value of **-1** = perfect negative relationship (as one goes up, the other goes down)
- Value of **0** = no relationship
- Value of **+1** = perfect positive relationship

We expect a **negative** correlation between load shedding hours and revenue.

In [ ]:
corr_hours = trading['hours_affected'].corr(trading['revenue'])
corr_stage = trading['stage'].corr(trading['revenue'])

print(f'Correlation: hours affected vs revenue : {corr_hours:.3f}')
print(f'Correlation: stage vs revenue          : {corr_stage:.3f}')
print()
if corr_hours < -0.5:
    print('✅ Strong negative correlation confirmed.')
    print('   Load shedding hours are strongly linked to lower revenue.')
elif corr_hours < -0.3:
    print('⚠️  Moderate negative correlation.')
else:
    print('ℹ️  Weak correlation — other factors dominate.')

## Step 2 — Linear Regression

**Linear regression** finds the best straight line through the data.
It learns a formula: `revenue = (coefficient × hours) + intercept`

The coefficient tells us: for every extra hour of load shedding, revenue changes by that amount.

In [ ]:
trading['is_weekend_int'] = trading['is_weekend'].astype(int)

X = trading[['hours_affected','is_weekend_int','month_num']].values
y = trading['revenue'].values

model = LinearRegression()
model.fit(X, y)

y_pred = model.predict(X)
r2  = r2_score(y, y_pred)
mae = mean_absolute_error(y, y_pred)

print('── Model Results ───────────────────────────────────')
print(f'R² Score          : {r2:.3f}')
print(f'Mean Absolute Error: R {mae:,.2f} per day')
print()
print('── Coefficients ────────────────────────────────────')
print(f'Each extra hour of LS reduces revenue by: R {abs(model.coef_[0]):,.2f}')
print(f'Weekend effect (extra revenue):           R {model.coef_[1]:,.2f}')
print(f'Month effect:                             R {model.coef_[2]:,.2f}')
print(f'Base daily revenue (intercept):           R {model.intercept_:,.2f}')

## Step 3 — What Does R² Mean?

R² (R-squared) measures what percentage of the variation in revenue is explained by our model.
- **R² = 1.0** → model explains 100% of variance (perfect)
- **R² = 0.6** → model explains 60% — a good result for real-world data
- **R² = 0.0** → model has no explanatory power

In [ ]:
print(f'R² = {r2:.3f}')
print(f'This means our model explains {r2*100:.1f}% of revenue variation.')
print()
print('The remaining variation is explained by other factors not in our data:')
print(' - Foot traffic patterns')
print(' - Local events or promotions')
print(' - Weather')
print(' - Stock availability')

## Step 4 — Scenario Forecasting for April 2024

Using the trained model, we predict daily revenue for April under four different load shedding scenarios.

In [ ]:
scenarios = {
    'No Load Shedding (Stage 0)':     [0,  0, 4],
    'Low Load Shedding (Stage 2)':    [4,  0, 4],
    'Medium Load Shedding (Stage 4)': [8,  0, 4],
    'High Load Shedding (Stage 6)':   [12, 0, 4],
}

results = []
for name, features in scenarios.items():
    daily = max(0, model.predict([features])[0])
    monthly = daily * 22  # assume 22 trading weekdays in April
    results.append({'Scenario': name, 'Daily (R)': daily, 'Monthly Est. (R)': monthly})

forecast_df = pd.DataFrame(results)
forecast_df['Daily (R)']    = forecast_df['Daily (R)'].map(lambda x: f'R {x:,.0f}')
forecast_df['Monthly Est. (R)'] = forecast_df['Monthly Est. (R)'].map(lambda x: f'R {x:,.0f}')
print(forecast_df.to_string(index=False))

In [ ]:
# Visualise the forecast
results_num = []
for name, features in scenarios.items():
    daily = max(0, model.predict([features])[0])
    results_num.append({'Scenario': name, 'Daily Revenue': daily})

fdf = pd.DataFrame(results_num)
fig = px.bar(fdf, x='Scenario', y='Daily Revenue',
             title='April 2024 Revenue Forecast by Load Shedding Scenario',
             labels={'Daily Revenue':'Predicted Daily Revenue (R)'},
             color='Daily Revenue',
             color_continuous_scale='RdYlGn',
             template='plotly_dark')
fig.update_layout(xaxis_tickangle=-15)
fig.show()

## Step 5 — Actual vs Predicted Revenue

Plotting actual revenue against what the model predicted for each day. A good model should have points clustered close to the diagonal line.

In [ ]:
trading['predicted'] = y_pred
fig = px.scatter(trading, x='revenue', y='predicted',
                 color='stage_category',
                 title='Actual vs Predicted Revenue',
                 labels={'revenue':'Actual Revenue (R)', 'predicted':'Predicted Revenue (R)'},
                 template='plotly_dark',
                 color_discrete_map={'None':'#2ecc71','Low (1-2)':'#f39c12',
                                      'Medium (3-4)':'#e67e22','High (5-6)':'#e74c3c'})
# Add diagonal perfect-prediction line
min_val = min(trading['revenue'].min(), trading['predicted'].min())
max_val = max(trading['revenue'].max(), trading['predicted'].max())
fig.add_trace(go.Scatter(x=[min_val,max_val], y=[min_val,max_val],
                          mode='lines', name='Perfect Prediction',
                          line=dict(color='white', dash='dash')))
fig.show()